# 第 6 章实验 · 时间序列：分解、平稳性、ACF、回测与 LSTM 外推

> 对应正文：[06-时间序列](../docs/4-专业方向/06-时间序列/README.md) ·
> [01 时间序列分析入门](../docs/4-专业方向/06-时间序列/01-时间序列分析入门.md) ·
> [02 RNN 与 LSTM](../docs/4-专业方向/06-时间序列/02-RNN与LSTM.md)

**环境**：实验 1~5 只需 `numpy / pandas / matplotlib / statsmodels`，全部离线可跑；
实验 6 的 LSTM 需要 `pip install torch`（本 cell 已注明本地预期，缺失 torch 时可跳过，不影响其余实验）。
数据全部由代码生成、随机种子固定——从上到下完整执行一遍即复现全部结果。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

np.random.seed(0)   # 固定随机种子，全 notebook 结果可复现

# 中文字体：Windows 优先微软雅黑/黑体，Mac/Linux 自动回退到检测到的字体
_available = {f.name for f in font_manager.fontManager.ttflist}
_cjk = [f for f in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
                    "PingFang SC", "WenQuanYi Micro Hei"] if f in _available]
plt.rcParams["font.sans-serif"] = _cjk + plt.rcParams["font.sans-serif"]
plt.rcParams["axes.unicode_minus"] = False    # 让负号正常显示
print("NumPy", np.__version__, "| pandas", pd.__version__, "| 中文字体:", _cjk)

## 实验一：手动加法分解——把序列拆成"大势 + 习气 + 抖动"

正文用 STL / `seasonal_decompose` 一键分解，这里手动复刻**加法分解**的三步：
中心化移动平均估计趋势 → 去趋势后按月平均估计季节 → 剩下的就是残差。
目标是看清 $Y_t = T_t + S_t + R_t$ 每一步到底在算什么。

In [ ]:
n = 120                                          # 10 年 × 12 个月
t = np.arange(n)
trend_true = 100 + 0.5 * t                       # 趋势：底座每月抬 0.5（每年 +6）
season_true = 20 * np.sin(2 * np.pi * t / 12)    # 季节：一年一个周期，振幅 ±20
noise = np.random.normal(0, 5, n)                # 残差：σ=5 的随机抖动
y = trend_true + season_true + noise             # 加法合成：观测 = 大势 + 习气 + 抖动
print("合成序列前 6 个月：", y[:6].round(1))

# 第 1 步：趋势 = 12 个月中心化移动平均（窗口对称，不引入相位偏移）
trend_est = pd.Series(y).rolling(12, center=True).mean()

# 第 2 步：季节 = 去趋势后的部分按"月份"分组求平均（12 个数，再平铺到每一年）
detrended = y - trend_est
month = np.tile(np.arange(12), n // 12)
season_est = np.array([np.nanmean(detrended[month == m]) for m in range(12)])
season_est -= season_est.mean()                  # 季节分量去均值，幅度不与趋势混淆
season_full = np.tile(season_est, n // 12)

# 第 3 步：残差 = 观测 − 趋势估计 − 季节估计
resid_est = y - trend_est - season_full
valid = ~np.isnan(resid_est)                     # 移动平均在首尾各缺 6 个点

print("估计的季节分量（1~12 月）：", season_est.round(1))
print("真实季节振幅 ±20，估计的波峰 %.1f / 波谷 %.1f" % (season_est.max(), season_est.min()))
print("与真实季节分量的相关系数 = %.3f" % np.corrcoef(season_full[valid], season_true[valid])[0, 1])
print("残差标准差 = %.2f（真实噪声 σ = 5.00）" % np.nanstd(resid_est))
# 预期输出：季节估计波峰 ~19.1 / 波谷 ~-21.5（真实 ±20）；与真值相关 ~0.99；
#           残差标准差 ~4.5——略小于真实 σ=5，因为移动平均顺带平滑掉了一点噪声

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
axes[0].plot(t, y, lw=0.9)
axes[0].set_title("加法分解：原序列 = 趋势 + 季节 + 残差")
axes[0].set_ylabel("观测值")
axes[1].plot(t, trend_true, "--", lw=1, color="gray", label="真实趋势")
axes[1].plot(t, trend_est, lw=1.6, label="移动平均估计")
axes[1].set_ylabel("趋势"); axes[1].legend()
axes[2].plot(t[:24], season_true[:24], "--o", ms=3, lw=1, color="gray", label="真实季节")
axes[2].plot(t[:24], season_full[:24], lw=1.2, label="月均值估计")
axes[2].set_ylabel("季节"); axes[2].legend()
axes[3].plot(t[valid], resid_est[valid], lw=0.8)
axes[3].axhline(0, color="red", ls=":", lw=0.8)
axes[3].set_ylabel("残差"); axes[3].set_xlabel("月份序号（0 = 第 1 年 1 月）")
plt.tight_layout(); plt.show()

**观察**：手动分解几乎完整还原了三股力量——季节分量与真值的相关系数 0.99 量级、
残差标准差约 4.5（略小于真实 σ=5：移动平均顺带平滑了一点噪声）。预测的力气应该花在
前两项上（趋势外推 + 季节修正），残差基本不可预测、留出容错即可——
这正是正文"望闻问切"第一步的数值版。

## 实验二：差分与平稳——随机游走 vs AR(1)（φ=0.5）的方差增长

正文"AR(1) 平稳性深潜"的两极：随机游走（φ=1）方差随 t **线性发散** $v_t = t\sigma^2$；
AR(1)（φ=0.5）方差收敛到平台 $\sigma^2/(1-\phi^2) \approx 1.33$。
这里模拟 2000 条路径，用每个时刻的**样本方差**亲眼验证——这是"涨潮的河 vs 水位稳定的河"的可跑版。

In [ ]:
rng = np.random.default_rng(42)
T, M = 200, 2000                       # 每条路径 200 步，共 2000 条路径
eps = rng.normal(0, 1, (M, T))         # 白噪声：两种过程共用同一批"踢力"

rw = np.cumsum(eps, axis=1)            # 随机游走（φ=1）：y_t = y_{t-1} + ε_t
ar = np.zeros((M, T))                  # AR(1)（φ=0.5）：y_t = 0.5·y_{t-1} + ε_t，从 0 起步
for k in range(1, T):
    ar[:, k] = 0.5 * ar[:, k - 1] + eps[:, k]

var_rw = rw.var(axis=0)                # 每个时刻 t 横跨 2000 条路径的样本方差
var_ar = ar.var(axis=0)
print("时刻 t    随机游走方差（经验 / 理论 t+1）     AR(1) 方差（经验 / 理论）")
for k in [0, 9, 49, 99, 199]:
    print(f"t={k:3d}      {var_rw[k]:7.1f} / {k + 1:6.1f}              "
          f"{var_ar[k]:5.3f} / {(1 - 0.25 ** k) / 0.75:5.3f}")
print("理论平台：σ²/(1−φ²) = 1/0.75 = %.3f；随机游走 200 步应涨到 ~200" % (1 / 0.75))
# 预期输出：随机游走经验方差紧贴 t+1（t=199 时 195.1 vs 200）；
#           AR(1) 从第 10 步起就稳定在 1.33~1.38 的平台——两条曲线一发散一收敛

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(var_rw, label="随机游走 φ=1（经验）", color="tab:red")
ax.plot(np.arange(1, T + 1), "--", color="tab:red", alpha=0.5, label="理论 t·σ²")
ax.plot(var_ar, label="AR(1) φ=0.5（经验）", color="tab:blue")
ax.axhline(1 / 0.75, ls=":", color="tab:blue", label="理论平台 σ²/(1−φ²)=1.33")
ax.set_yscale("log")
ax.set_title("方差增长：随机游走线性发散 vs AR(1) 收敛到平台（对数纵轴）")
ax.set_xlabel("时刻 t"); ax.set_ylabel("方差（对数刻度）"); ax.legend()
plt.tight_layout(); plt.show()

**观察**：AR(1) 的方差几步之内就贴住 1.33 的平台——"有回拉力的弹簧"把每一步的踢力都吸收掉；
随机游走的方差一路按 t·σ² 累计（200 步 ≈ 200）。统计模型要的"平稳入场券"，
本质就是要这条方差曲线有水平渐近线；差分（ARIMA 的 I）干的就是把 φ=1 换成 φ=0 的增量。

## 实验三：AR(1) 方差递推——φ = 0.5 / 0.95 / 0.99 的三条收敛曲线

不模拟、纯递推：$v_t = \phi^2 v_{t-1} + \sigma^2$（$v_0=0$，$\sigma^2=1$），
极限 $v_\infty = \sigma^2/(1-\phi^2)$。这是正文深潜"小算例 + 三条方差轨迹图"的可跑版：
φ 越接近 1，极限方差越大、收敛越慢——φ=0.99 就是"假平静"（有限样本里与随机游走肉眼难分）。

In [ ]:
sigma2, T = 1.0, 400
phis = [0.5, 0.95, 0.99]
v = {p: [0.0] for p in phis}                     # v_0 = 0
for p in phis:
    for _ in range(T):
        v[p].append(p ** 2 * v[p][-1] + sigma2)  # 递推：v_t = φ²·v_{t-1} + σ²
v = {p: np.array(x) for p, x in v.items()}

print("φ      极限 σ²/(1−φ²)   v[5]    v[50]   v[100]   v[400]")
for p in phis:
    vinf = sigma2 / (1 - p ** 2)
    print(f"{p:.2f}      {vinf:6.2f}       {v[p][5]:6.2f}  {v[p][50]:6.2f}  "
          f"{v[p][100]:6.2f}  {v[p][400]:6.2f}")
print("对照 φ=1（随机游走）：v_t = t·σ² → v[100] =", 100 * sigma2)
# 预期输出：φ=0.50 五步到位（v[5]=1.33，正文小算例同款）；φ=0.95 到 v[50]=10.20 才接近极限
#           10.26；φ=0.99 极限 50.25、v[100] 才 43.5、v[400] 才贴住极限——收敛上千步

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = {0.5: "tab:blue", 0.95: "tab:orange", 0.99: "tab:green"}
for p in phis:
    ax.plot(v[p], color=colors[p], label=f"φ={p}（极限 {1/(1-p**2):.1f}）")
    ax.axhline(sigma2 / (1 - p ** 2), ls=":", color=colors[p], alpha=0.6)
ax.plot(np.arange(T + 1) * sigma2, lw=1, color="tab:red", alpha=0.7,
        label="φ=1（线性发散）")
ax.set_yscale("log")
ax.set_title("AR(1) 方差递推：|φ|<1 有水平渐近线，φ→1 收敛越来越慢")
ax.set_xlabel("时刻 t"); ax.set_ylabel("方差 v_t（对数刻度）"); ax.legend()
plt.tight_layout(); plt.show()

**观察**：φ=0.5 几步到位（v[5]=1.33，正文小算例同款数字）；φ=0.95 爬到 t=50 才 10.2；
φ=0.99 极限方差 50、v[100] 才 43.5、收敛要上千步。**平稳 = 方差轨迹有水平渐近线**；
ADF 检验检验的正是"有没有这条渐近线"，而它在近单位根处功效低——φ=0.99 这条曲线就是定量注脚。

## 实验四：ACF 手算——小序列的自相关 + statsmodels 对照

先用正文"举个例子"的 8 点爬坡序列 $y=[3,5,4,6,5,7,6,8]$ 手算 ACF：
$$\rho_k = \frac{\sum_t (y_t-\bar y)(y_{t-k}-\bar y)}{\sum_t (y_t-\bar y)^2}$$
再用 statsmodels 的 `acf` / `plot_acf` 复核；最后造一条 AR(1)（φ=0.8）长序列，
看样本 ACF 是否还原理论值 $\rho_k = \phi^k$ 的指数拖尾。

In [ ]:
y_small = np.array([3, 5, 4, 6, 5, 7, 6, 8], dtype=float)   # 正文的小序列
ybar = y_small.mean()
dev = y_small - ybar
denom = np.sum(dev ** 2)                       # 分母：滞后 0 的"总动能"
print(f"均值 ȳ = {ybar:.2f}，分母 Σ(y−ȳ)² = {denom:.2f}")
for k in [1, 2, 3]:
    num = np.sum(dev[k:] * dev[:-k])           # 分子：y 与错开 k 步的自己逐项相乘
    print(f"ACF({k}) = {num / denom:+.3f}   （分子 = {num:+.2f}）")
# 预期输出：ACF(1)=+0.125、ACF(2)=+0.472、ACF(3)=−0.236——
#           滞后 2 反而最高：这条"涨两步退一步"的爬坡序列每隔两步最像自己

In [ ]:
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf

sm_vals = acf(y_small, nlags=3)
print("statsmodels ACF(1~3)：", sm_vals[1:].round(3), "← 应与手算完全一致")

rng = np.random.default_rng(7)
n_sim, phi = 500, 0.8
ar_series = np.zeros(n_sim)
for k in range(1, n_sim):                      # 造一条足够长的 AR(1) 序列
    ar_series[k] = phi * ar_series[k - 1] + rng.normal()

lags = np.arange(1, 11)
print("滞后 k    ：", lags)
print("样本 ACF  ：", acf(ar_series, nlags=10)[1:].round(3))
print("理论 φ^k  ：", (phi ** lags).round(3))
# 预期输出：statsmodels 与手算逐位一致；样本 ACF 从 ~0.79 起沿 0.8^k 指数衰减（拖尾），
#           长滞后处略高于理论——滞后越接近样本长度，样本 ACF 越不可信

fig, ax = plt.subplots(figsize=(7, 4))
plot_acf(ar_series, lags=10, ax=ax)
ax.set_title("AR(1) φ=0.8 序列的 ACF：指数拖尾（蓝色区域为 95% 置信带）")
ax.set_xlabel("滞后阶数 k"); ax.set_ylabel("自相关系数")
plt.tight_layout(); plt.show()

**观察**：手算与 statsmodels 逐位一致；AR(1) 的样本 ACF 沿 $\phi^k$ 指数衰减（拖尾）。
这正是定阶口诀"**AR 看 PACF 截尾、MA 看 ACF 截尾**"的前半句现场——
ACF 拖得越长，惯性越长，可预测的"本钱"越多。

## 实验五：三个朴素基线 + 滚动回测——时序评估严禁 shuffle

造 10 年月度序列，比较三个基线：**朴素**（重复最后一个值）、**历史均值**、**线性趋势外推**。
评估用**滚动回测**（rolling origin）：训练窗从 6 年起每次向前滚 12 个月、预测未来 12 个月，
报告 MAPE（时序爱用百分比误差——"错 10 台"对日销 100 和 1 万的店意义完全不同）。
切分严格按时间先后，绝不随机打乱——否则"未来"混进训练集，是时序项目最隐蔽的泄漏。

In [ ]:
n = 120
t = np.arange(n)
y5 = 200 + 0.8 * t + 15 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 8, n)

def mape(pred, true):
    return np.mean(np.abs((true - pred) / true)) * 100   # 平均绝对百分比误差

def run_baselines(train, horizon=12):
    last = train[-1]
    idx = np.arange(len(train))
    k, b = np.polyfit(idx, train, 1)         # 最小二乘拟合线性趋势
    fut = np.arange(len(train), len(train) + horizon)
    return {
        "朴素（重复最后值）": np.full(horizon, last),
        "历史均值":          np.full(horizon, train.mean()),
        "线性趋势外推":      k * fut + b,
    }

rows, folds = [], []
for fi, start in enumerate(range(72, n - 12 + 1, 12), 1):   # 滚动回测：4 折，绝不 shuffle
    train, test = y5[:start], y5[start:start + 12]
    folds.append(f"折{fi}·起点{start}")
    for name, pred in run_baselines(train).items():
        rows.append({"切分": f"折{fi}·起点{start}",
                     "基线": name, "MAPE(%)": round(mape(pred, test), 2)})
bt = pd.DataFrame(rows)
pivot = bt.pivot(index="基线", columns="切分", values="MAPE(%)")[folds]
pivot["平均"] = pivot.mean(axis=1).round(2)
print(pivot.to_string())
# 预期输出：线性趋势外推平均 ~3.8%，朴素 ~4.8%，历史均值 ~14.8%——
#           历史均值完全跟不上爬坡；基线之间能差 4 倍，基线选错=参照系全错

**观察**：线性趋势基线在所有切分上明显占优，历史均值最差——**基线选错了，后面的复杂模型
连参照系都没有**。另外注意滚动回测的铁律：每次切分都保证"训练全部早于测试"，
与第 3 章的 K 折交叉验证（随机 shuffle）正好相反——时序里 shuffle 等于把答案塞进考卷。

## 实验六：LSTM 外推正弦（需 torch）+ numpy 指数平滑对照版

正文 02 页的招牌实验：训练段只给前 140 个正弦点，LSTM 学会后**自回归外推** 60 步
（预测值塞回输入窗口继续预测）仍能接上周期。下面第一个 cell 需要 `pip install torch`
（离线最小环境没有 torch 时直接跳过，不影响其它实验）；第二个 cell 是 **numpy 指数平滑
替代版**，任何环境可跑——平滑系数 α 与 LSTM 遗忘门在"新旧信息按比例混合"上同构：
前者是人工固定的比例，后者是学出来的逐维阀门（正文"深度视角"）。

In [ ]:
# 依赖：pip install torch（需本地环境：本仓库的离线验证机器装了 torch 但 DLL 加载失败，
# 此 cell 仅通过语法校验；预期输出来自正文 02 页同款实验的实跑经验，装好 torch 后即可复现）
import torch
import torch.nn as nn

torch.manual_seed(0)
t_axis = np.linspace(0, 12 * np.pi, 200)
ysin = np.sin(t_axis).astype(np.float32)
SEQ = 20                                          # 每次读连续 20 个点，预测第 21 个
X = torch.tensor(np.stack([ysin[i:i + SEQ] for i in range(140 - SEQ)])).unsqueeze(-1)
Y = torch.tensor(ysin[SEQ:140]).unsqueeze(-1)
print("训练样本形状：", tuple(X.shape))            # 120 个样本 × 20 步 × 1 维

class LSTMRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=32, batch_first=True)
        self.head = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.lstm(x)                     # out：每一步的隐藏状态 [批, 20, 32]
        return self.head(out[:, -1])              # 只用最后一步——整段输入的记忆浓缩

model = LSTMRegressor()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
for epoch in range(301):                          # 全批量训练（CPU 约半分钟）
    loss = loss_fn(model(X), Y)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)   # 梯度裁剪防爆炸
    opt.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.6f}")

model.eval()
window, preds = list(ysin[140 - SEQ:140]), []
with torch.no_grad():                             # 自回归外推 60 步 ≈ 1.8 个周期
    for _ in range(60):
        x = torch.tensor(window[-SEQ:]).view(1, SEQ, 1)
        nxt = model(x).item()
        preds.append(nxt); window.append(nxt)
err = np.abs(np.array(preds) - np.sin(t_axis[140:]))
print(f"外推 60 步平均绝对误差 = {err.mean():.3f}（模型从没见过第 140 步之后的点）")
# 预期输出（正文 02 页同款实验的实跑经验，随种子略有浮动）：
#   loss 从 ~0.5 量级降到 1e-4~1e-3；外推 60 步平均绝对误差多在 0.1~0.2——
#   模型没见过第 140 步之后的任何点，却能接上周期规律，这就是"学到了周期记忆"

In [ ]:
# numpy 替代版：一阶指数平滑 ŷ_{t+1} = α·y_t + (1−α)·ŷ_t（无需任何额外依赖，离线必跑）
t_axis = np.linspace(0, 12 * np.pi, 200)         # 与 torch cell 相同的正弦设置（自包含）
train_sine = np.sin(t_axis[:140])

def es_one_step(series, alpha):
    level, errs = series[0], []                   # 平滑值（水平项）从第一个点起步
    for v in series[1:]:                          # 每步先用当前水平预测下一个点，再更新水平
        errs.append(abs(level - v))
        level = alpha * v + (1 - alpha) * level
    return np.array(errs), level

def es_extrapolate(level, horizon=60):
    return np.full(horizon, level)                # 无趋势/无季节项：外推 = 水平的常数延伸

print("α     单步预测 MAE   外推 60 步 MAE")
for alpha in [0.1, 0.3, 0.7]:
    errs1, level = es_one_step(train_sine, alpha)
    mae_ext = np.abs(es_extrapolate(level) - np.sin(t_axis[140:])).mean()
    print(f"{alpha:.1f}       {errs1.mean():.3f}          {mae_ext:.3f}")
# 预期输出：α 越大单步 MAE 越小（0.1→0.58，0.7→0.17，跟得越紧），
#           但外推 60 步全部在 0.65 以上——外推只剩一条水平线，周期信息全丢

_, level03 = es_one_step(train_sine, 0.3)
plt.figure(figsize=(8, 3.5))
plt.plot(t_axis[130:], np.sin(t_axis[130:]), color="gray", alpha=0.8, label="真实正弦")
plt.plot(t_axis[140:], es_extrapolate(level03), "--", lw=2, label="指数平滑外推（α=0.3）")
plt.axvline(t_axis[140], color="red", ls=":", label="训练段结束位置")
plt.title("指数平滑外推正弦：拉平成一条水平线（记不住周期）")
plt.xlabel("t"); plt.ylabel("y"); plt.legend()
plt.tight_layout(); plt.show()

**观察**：指数平滑的**单步**预测不算差（正弦平滑、惯性大，α=0.7 时单步 MAE≈0.17），
但**外推**立刻露馅——没有周期项的它把未来 60 步全部预测成同一条水平线（外推 MAE 0.66~0.98，
α 越大反而越差）。对照 torch cell 的 LSTM：正文同款实验实跑的外推平均绝对误差在 **0.1~0.2** 量级——
"预测值塞回去继续预测"这道硬测试下，单步拟合好 ≠ 学到了结构；
LSTM 把周期"记忆"进了隐藏状态，指数平滑只记住了一个水平。

## 改参数建议（一次只改一个，观察一个）

1. **实验一**：把季节振幅改成随水平放大（如 `season = 0.15 * trend_true * sin(...)`，即乘法序列），再按乘法思路**先取对数、再做加法分解**——体会加法/乘法分解的选择依据；
2. **实验三**：φ 改成 0.999，看极限方差（500σ²）与收敛所需步数怎么变——"近单位根的假平静"有多假；
3. **实验四**：把 φ 改成 −0.8（符号振荡），ACF 变成正负交替的拖尾——平稳条件是 |φ|<1 这个**绝对值**不等式；
4. **实验五**：给 `run_baselines` 加一个"季节朴素"基线（用去年同月值预测今年同月），MAPE 通常再降一档——季节信息值多少钱，一看便知；
5. **实验六（torch cell）**：学习率 1e-2 改 1e-1 观察 loss 发散（梯度爆炸现场），再把 SEQ 20 改 60 体会长依赖的难度台阶；或把 α 从 0.1 扫到 0.95，看指数平滑"单步更灵敏、外推依旧拉平"的不对称。